<a href="https://colab.research.google.com/github/FizaAslam1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FizaAslam1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:

# Apna token directly paste karo
HF_TOKEN = ""

from huggingface_hub import login
login(token=HF_TOKEN)
print("✅ Hugging Face login successful!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face login successful!


In [14]:
from huggingface_hub import login
import duckdb
import pandas as pd

# Paste your token
HF_TOKEN = ""
login(token=HF_TOKEN)

con = duckdb.connect()

# Install and load httpfs extension
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Set the token
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

# Now load data
query = """
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df = con.execute(query).df()
print(f"✅ Data loaded! Shape: {df.shape}")
df.head()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Data loaded! Shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [17]:
# ==========================================
# 1. UNIT OF ANALYSIS + TIME WINDOW
# ==========================================
print("=" * 60)
print("1. UNIT OF ANALYSIS + TIME WINDOW")
print("=" * 60)

# Statement
print("""
One row = One content page's daily performance metrics.
Time window = March 2026 (2026-03-01 to 2026-03-31)
Grain = (report_date, content_hash_id)
""")

# Verify — Check unique combinations
grain_check = df.groupby(['report_date', 'content_hash_id']).size()
print(f"\n🔍 Verification:")
print(f"Total rows: {len(df):,}")
print(f"Unique (date, page) combinations: {len(grain_check):,}")
print(f"One row = one page per day? {len(df) == len(grain_check)}")

# Show date range
print(f"\n📅 Date range: {df['report_date'].min()} to {df['report_date'].max()}")
print(f"📅 Unique dates: {df['report_date'].nunique()}")
print(f"📄 Unique pages: {df['content_hash_id'].nunique()}")
print(f"🏢 Unique clients: {df['client_hash_id'].nunique()}")

1. UNIT OF ANALYSIS + TIME WINDOW

One row = One content page's daily performance metrics.
Time window = March 2026 (2026-03-01 to 2026-03-31)
Grain = (report_date, content_hash_id)


🔍 Verification:
Total rows: 9,841,378
Unique (date, page) combinations: 9,841,378
One row = one page per day? True

📅 Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
📅 Unique dates: 31
📄 Unique pages: 331437
🏢 Unique clients: 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================
# 2. FIELDS CLASSIFICATION
# ==========================================
print("\n" + "=" * 60)
print("2. FIELDS: FEATURE / LABEL / CONTEXT / EXCLUDED")
print("=" * 60)

print("""
📊 FEATURES (Knowable at decision moment):
  1. gsc_impressions — past visibility signal
  2. gsc_clicks — past engagement signal
  3. gsc_sum_position — search ranking quality
  4. ctr (calculated: clicks/impressions) — click-through rate
  5. report_date — temporal context

🏷️ LABEL (What we want to predict):
  → Content needs refresh? (Derived from declining trends)
  → Will use trend_direction from dim_content table (down = needs refresh)

📋 CONTEXT (Not features, but useful):
  - client_hash_id — grouping
  - content_hash_id — identification
  - gsc_data_available — filter flag
  - ga4_data_available — filter flag

❌ EXCLUDED (With reasons):
  - ga4_* columns → Not all clients have GA4, sparse data
  - ai_* columns → AI traffic is separate signal, not for refresh decisions
  - scroll_events → Not relevant for content refresh prioritization
  - month → Same as report_date month, redundant
""")


2. FIELDS: FEATURE / LABEL / CONTEXT / EXCLUDED

📊 FEATURES (Knowable at decision moment):
  1. gsc_impressions — past visibility signal
  2. gsc_clicks — past engagement signal  
  3. gsc_sum_position — search ranking quality
  4. ctr (calculated: clicks/impressions) — click-through rate
  5. report_date — temporal context

🏷️ LABEL (What we want to predict):
  → Content needs refresh? (Derived from declining trends)
  → Will use trend_direction from dim_content table (down = needs refresh)

📋 CONTEXT (Not features, but useful):
  - client_hash_id — grouping
  - content_hash_id — identification
  - gsc_data_available — filter flag
  - ga4_data_available — filter flag

❌ EXCLUDED (With reasons):
  - ga4_* columns → Not all clients have GA4, sparse data
  - ai_* columns → AI traffic is separate signal, not for refresh decisions
  - scroll_events → Not relevant for content refresh prioritization
  - month → Same as report_date month, redundant



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================
# 3. VERIFY WITH QUERIES
# ==========================================
print("\n" + "=" * 60)
print("3. VERIFICATION QUERIES")
print("=" * 60)

# Query 1: Grain verification
print("\n📊 QUERY 1: Grain Check")
print(f"   Statement: One row = one page per day")
print(f"   Result: {len(df)} rows = {len(grain_check)} unique (date,page) pairs")
print(f"   Verdict: {'✅ PASS' if len(df) == len(grain_check) else '❌ FAIL'}")

# Query 2: Count verification
print("\n📊 QUERY 2: Row Count & Date Span")
print(f"   Total rows: {len(df):,}")
print(f"   Date span: {df['report_date'].min()} to {df['report_date'].max()}")
print(f"   Days: {df['report_date'].nunique()}")
print(f"   Pages: {df['content_hash_id'].nunique():,}")
print(f"   Clients: {df['client_hash_id'].nunique():,}")

# Query 3: Availability — GSC data with IS TRUE
print("\n📊 QUERY 3: Data Availability")
total_rows = len(df)
gsc_available = df[df['gsc_data_available'] == True]
gsc_unavailable = df[df['gsc_data_available'] != True]

print(f"   Rows with GSC available: {len(gsc_available):,} ({len(gsc_available)/total_rows*100:.1f}%)")
print(f"   Rows without GSC: {len(gsc_unavailable):,} ({len(gsc_unavailable)/total_rows*100:.1f}%)")

# Missing values check
print(f"\n📊 Missing Values (key columns):")
for col in ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position']:
    missing = df[col].isna().sum()
    print(f"   {col}: {missing:,} missing ({missing/total_rows*100:.1f}%)")

# Query 4: Non-zero impressions
has_impressions = df[(df['gsc_data_available'] == True) & (df['gsc_impressions'] > 0)]
print(f"\n📊 Pages with impressions > 0: {len(has_impressions):,}")
print(f"   Unique pages with impressions: {has_impressions['content_hash_id'].nunique():,}")


3. VERIFICATION QUERIES

📊 QUERY 1: Grain Check
   Statement: One row = one page per day
   Result: 9841378 rows = 9841378 unique (date,page) pairs
   Verdict: ✅ PASS

📊 QUERY 2: Row Count & Date Span
   Total rows: 9,841,378
   Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
   Days: 31
   Pages: 331,437
   Clients: 55

📊 QUERY 3: Data Availability
   Rows with GSC available: 3,611,061 (36.7%)
   Rows without GSC: 6,230,317 (63.3%)

📊 Missing Values (key columns):
   gsc_impressions: 0 missing (0.0%)
   gsc_clicks: 0 missing (0.0%)
   gsc_sum_position: 0 missing (0.0%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================
# 4. DATA LIMITS
# ==========================================
print("\n" + "=" * 60)
print("4. DATA LIMITS")
print("=" * 60)

print("""
⚠️ WHAT THIS DATA CAN NEVER TELL YOU:

1. Unbalanced History:
   - Some clients joined late → incomplete historical data
   - New pages have less history than old ones
   - Cannot predict for pages with <30 days of data

2. GSC-Only Early Rows:
   - GA4 data is sparse (many NULLs)
   - Early months may only have GSC, no GA4
   - Cannot use engagement metrics for all pages

3. Window Overlaps:
   - Monthly aggregates may hide weekly patterns
   - A page declining in week 1 might recover by week 4
   - Month-level data loses granularity

4. Client Differences:
   - Clients in different industries behave differently
   - Cannot generalize one client's patterns to another
   - Model may not transfer across client domains

5. Causation vs Correlation:
   - Can observe decline but cannot explain WHY
   - Algorithm updates, seasonality, competitor changes — all invisible
   - We predict correlation, not causation

6. Limited Label:
   - trend_direction is a proxy, not ground truth
   - A "declining" page might still be valuable
   - Content refresh might not fix all declining pages
""")

print("✅ Data contract complete!")


4. DATA LIMITS

⚠️ WHAT THIS DATA CAN NEVER TELL YOU:

1. Unbalanced History:
   - Some clients joined late → incomplete historical data
   - New pages have less history than old ones
   - Cannot predict for pages with <30 days of data

2. GSC-Only Early Rows:
   - GA4 data is sparse (many NULLs)
   - Early months may only have GSC, no GA4
   - Cannot use engagement metrics for all pages

3. Window Overlaps:
   - Monthly aggregates may hide weekly patterns
   - A page declining in week 1 might recover by week 4
   - Month-level data loses granularity

4. Client Differences:
   - Clients in different industries behave differently
   - Cannot generalize one client's patterns to another
   - Model may not transfer across client domains

5. Causation vs Correlation:
   - Can observe decline but cannot explain WHY
   - Algorithm updates, seasonality, competitor changes — all invisible
   - We predict correlation, not causation

6. Limited Label:
   - trend_direction is a proxy, not ground 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.